# What can Machine Learning tell us about US Air Pollution?

We'll take the CSV output from the Visualisations notebook and look to see what an ML model can glean from the data

In [10]:
import pandas as pd
import numpy as np
import re
import sklearn


--------

### Step 1 - import data; strip out unnecessary pollution data, align location data across both files, join files.

There are 3 steps we're going to take to make our data better shaped for machine learning:

1) remove any data before 2010, as we want our analysis to be relevant and not outdated
2) ensure that both data files have a common key, in order to join them; because that common key will include a 2char state code, we'll remove any data relating to Mexico
3) join the data based on that common location key.

In [16]:
# Load the dataset with state codes
poll = pd.read_csv("cleaned_pollution_data_with_state_codes.csv")

# Split the combined field
parts = poll["City_County_State"].str.split(",", expand=True)

poll["County"] = parts[1].str.strip()

poll.head()


,Unnamed: 0,State,Date Local,NO2 Mean (ppb),NO2 1st Max Value (ppb),NO2 1st Max Hour,NO2 AQI,O3 Mean (ppm),O3 1st Max Value (ppm),O3 1st Max Hour,...,CO_AQI_imp,NO2_AQI_imp,Year,Quarter,DayOfWeek,City_County_State,City,AreaType,StateCode,County
0,0,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,47.73,17.96,2000,1,Saturday,"Phoenix, Maricopa, Arizona",Phoenix,Urban,AZ,Maricopa
1,0,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,47.73,17.96,2000,1,Saturday,"Phoenix, Maricopa, Arizona",Phoenix,Urban,AZ,Maricopa
2,1,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,25.00,17.96,2000,1,Saturday,"Phoenix, Maricopa, Arizona",Phoenix,Urban,AZ,Maricopa
3,1,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,25.00,17.96,2000,1,Saturday,"Phoenix, Maricopa, Arizona",Phoenix,Urban,AZ,Maricopa
4,2,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,47.73,17.96,2000,1,Saturday,"Phoenix, Maricopa, Arizona",Phoenix,Urban,AZ,Maricopa


In [24]:
poll.shape

(2333496, 32)

In [21]:
# join the county and statecode fields to create a new key field for merging
poll['County_StateCode'] = poll['County'] + ', ' + poll['StateCode']
poll.head()

,Unnamed: 0,State,Date Local,NO2 Mean (ppb),NO2 1st Max Value (ppb),NO2 1st Max Hour,NO2 AQI,O3 Mean (ppm),O3 1st Max Value (ppm),O3 1st Max Hour,...,NO2_AQI_imp,Year,Quarter,DayOfWeek,City_County_State,City,AreaType,StateCode,County,County_StateCode
0,0,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,17.96,2000,1,Saturday,"Phoenix, Maricopa, Arizona",Phoenix,Urban,AZ,Maricopa,"Maricopa, AZ"
1,0,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,17.96,2000,1,Saturday,"Phoenix, Maricopa, Arizona",Phoenix,Urban,AZ,Maricopa,"Maricopa, AZ"
2,1,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,17.96,2000,1,Saturday,"Phoenix, Maricopa, Arizona",Phoenix,Urban,AZ,Maricopa,"Maricopa, AZ"
3,1,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,17.96,2000,1,Saturday,"Phoenix, Maricopa, Arizona",Phoenix,Urban,AZ,Maricopa,"Maricopa, AZ"
4,2,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,17.96,2000,1,Saturday,"Phoenix, Maricopa, Arizona",Phoenix,Urban,AZ,Maricopa,"Maricopa, AZ"


In [17]:
# now load the demographics data and align the location fields
demo = pd.read_csv('../county_demographics.csv')   
demo.head()

,County,State,Age.Percent 65 and Older,Age.Percent Under 18 Years,Age.Percent Under 5 Years,Education.Bachelor's Degree or Higher,Education.High School or Higher,Employment.Nonemployer Establishments,Ethnicities.American Indian and Alaska Native Alone,Ethnicities.Asian Alone,...,Population.Population per Square Mile,Sales.Accommodation and Food Services Sales,Sales.Retail Sales,Employment.Firms.Total,Employment.Firms.Women-Owned,Employment.Firms.Men-Owned,Employment.Firms.Minority-Owned,Employment.Firms.Nonminority-Owned,Employment.Firms.Veteran-Owned,Employment.Firms.Nonveteran-Owned
0,Abbeville County,SC,22.4,19.8,4.7,15.6,81.7,1416,0.3,0.4,...,51.8,12507,91371,1450,543,689,317,1080,187,1211
1,Acadia Parish,LA,15.8,25.8,6.9,13.3,79.0,4533,0.4,0.3,...,94.3,52706,602739,4664,1516,2629,705,3734,388,4007
2,Accomack County,VA,24.6,20.7,5.6,19.5,81.5,2387,0.7,0.8,...,73.8,53568,348195,2997,802,1716,335,2560,212,2536
3,Ada County,ID,14.9,23.2,5.6,38.5,95.2,41464,0.8,2.7,...,372.8,763099,5766679,41789,14661,19409,3099,36701,3803,35132
4,Adair County,IA,23.0,21.8,5.6,18.5,94.2,609,0.3,0.5,...,13.5,-1,63002,914,304,499,0,861,185,679


In [20]:
#remove the suffix 'county' from the County field in demo
demo['County'] = demo['County'].str.replace(r'\s*County\s*$', '', regex=True).str.strip()
demo.head()


,County,State,Age.Percent 65 and Older,Age.Percent Under 18 Years,Age.Percent Under 5 Years,Education.Bachelor's Degree or Higher,Education.High School or Higher,Employment.Nonemployer Establishments,Ethnicities.American Indian and Alaska Native Alone,Ethnicities.Asian Alone,...,Population.Population per Square Mile,Sales.Accommodation and Food Services Sales,Sales.Retail Sales,Employment.Firms.Total,Employment.Firms.Women-Owned,Employment.Firms.Men-Owned,Employment.Firms.Minority-Owned,Employment.Firms.Nonminority-Owned,Employment.Firms.Veteran-Owned,Employment.Firms.Nonveteran-Owned
0,Abbeville,SC,22.4,19.8,4.7,15.6,81.7,1416,0.3,0.4,...,51.8,12507,91371,1450,543,689,317,1080,187,1211
1,Acadia Parish,LA,15.8,25.8,6.9,13.3,79.0,4533,0.4,0.3,...,94.3,52706,602739,4664,1516,2629,705,3734,388,4007
2,Accomack,VA,24.6,20.7,5.6,19.5,81.5,2387,0.7,0.8,...,73.8,53568,348195,2997,802,1716,335,2560,212,2536
3,Ada,ID,14.9,23.2,5.6,38.5,95.2,41464,0.8,2.7,...,372.8,763099,5766679,41789,14661,19409,3099,36701,3803,35132
4,Adair,IA,23.0,21.8,5.6,18.5,94.2,609,0.3,0.5,...,13.5,-1,63002,914,304,499,0,861,185,679


In [22]:
#create a similar key field in demo for merging based on county and state field: give it the same name as in poll
demo['County_StateCode'] = demo['County'] + ', ' + demo['State']
demo.head()

,County,State,Age.Percent 65 and Older,Age.Percent Under 18 Years,Age.Percent Under 5 Years,Education.Bachelor's Degree or Higher,Education.High School or Higher,Employment.Nonemployer Establishments,Ethnicities.American Indian and Alaska Native Alone,Ethnicities.Asian Alone,...,Sales.Accommodation and Food Services Sales,Sales.Retail Sales,Employment.Firms.Total,Employment.Firms.Women-Owned,Employment.Firms.Men-Owned,Employment.Firms.Minority-Owned,Employment.Firms.Nonminority-Owned,Employment.Firms.Veteran-Owned,Employment.Firms.Nonveteran-Owned,County_StateCode
0,Abbeville,SC,22.4,19.8,4.7,15.6,81.7,1416,0.3,0.4,...,12507,91371,1450,543,689,317,1080,187,1211,"Abbeville, SC"
1,Acadia Parish,LA,15.8,25.8,6.9,13.3,79.0,4533,0.4,0.3,...,52706,602739,4664,1516,2629,705,3734,388,4007,"Acadia Parish, LA"
2,Accomack,VA,24.6,20.7,5.6,19.5,81.5,2387,0.7,0.8,...,53568,348195,2997,802,1716,335,2560,212,2536,"Accomack, VA"
3,Ada,ID,14.9,23.2,5.6,38.5,95.2,41464,0.8,2.7,...,763099,5766679,41789,14661,19409,3099,36701,3803,35132,"Ada, ID"
4,Adair,IA,23.0,21.8,5.6,18.5,94.2,609,0.3,0.5,...,-1,63002,914,304,499,0,861,185,679,"Adair, IA"


In [23]:
# Merge the datasets on the new key where pollution data is the left table and demo data is pulled in where available
merged = pd.merge(poll, demo, on='County_StateCode', how='left')    
merged.head()

,Unnamed: 0,State_x,Date Local,NO2 Mean (ppb),NO2 1st Max Value (ppb),NO2 1st Max Hour,NO2 AQI,O3 Mean (ppm),O3 1st Max Value (ppm),O3 1st Max Hour,...,Population.Population per Square Mile,Sales.Accommodation and Food Services Sales,Sales.Retail Sales,Employment.Firms.Total,Employment.Firms.Women-Owned,Employment.Firms.Men-Owned,Employment.Firms.Minority-Owned,Employment.Firms.Nonminority-Owned,Employment.Firms.Veteran-Owned,Employment.Firms.Nonveteran-Owned
0,0,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,414.9,9105877.0,57296663.0,324146.0,115146.0,163095.0,83284.0,227870.0,29940.0,278278.0
1,0,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,414.9,9105877.0,57296663.0,324146.0,115146.0,163095.0,83284.0,227870.0,29940.0,278278.0
2,1,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,414.9,9105877.0,57296663.0,324146.0,115146.0,163095.0,83284.0,227870.0,29940.0,278278.0
3,1,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,414.9,9105877.0,57296663.0,324146.0,115146.0,163095.0,83284.0,227870.0,29940.0,278278.0
4,2,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,414.9,9105877.0,57296663.0,324146.0,115146.0,163095.0,83284.0,227870.0,29940.0,278278.0


In [ ]:
#check that the merged table has the expected number of rows (same as poll table)
merged.shape

(2333496, 75)

In [27]:
merged.columns

Index(['Unnamed: 0', 'State_x', 'Date Local', 'NO2 Mean (ppb)',
       'NO2 1st Max Value (ppb)', 'NO2 1st Max Hour', 'NO2 AQI',
       'O3 Mean (ppm)', 'O3 1st Max Value (ppm)', 'O3 1st Max Hour', 'O3 AQI',
       'SO2 Mean (ppb)', 'SO2 1st Max Value (ppb)', 'SO2 1st Max Hour',
       'SO2 AQI', 'CO Mean (ppm)', 'CO 1st Max Value (ppm)', 'CO 1st Max Hour',
       'CO AQI', 'O3_AQI_imp', 'SO2_AQI_imp', 'CO_AQI_imp', 'NO2_AQI_imp',
       'Year', 'Quarter', 'DayOfWeek', 'City_County_State', 'City', 'AreaType',
       'StateCode', 'County_x', 'County_StateCode', 'County_y', 'State_y',
       'Age.Percent 65 and Older', 'Age.Percent Under 18 Years',
       'Age.Percent Under 5 Years', 'Education.Bachelor's Degree or Higher',
       'Education.High School or Higher',
       'Employment.Nonemployer Establishments',
       'Ethnicities.American Indian and Alaska Native Alone',
       'Ethnicities.Asian Alone', 'Ethnicities.Black Alone',
       'Ethnicities.Hispanic or Latino',
       'Ethnic

We now need to clean up some of the merged table coumns:
- remove newly duplicated location data
- consolidate data at county-state-year level

In [ ]:
# Keep the demographics county/state
merged = merged.drop(columns=[
    "County_x", 
    "State_x", 
    "StateCode", 
    "County_StateCode"
])

# Rename County_y and State_y to clean names
merged = merged.rename(columns={
    "County_y": "County",
    "State_y": "State"
})

KeyError: "['County_x', 'State_x', 'StateCode', 'County_StateCode'] not found in axis"

In [36]:
# aggregate data by county-state-year to get average pollution levels per year

pollutant_cols = [
    'NO2 Mean (ppb)', 'NO2 AQI',
    'O3 Mean (ppm)', 'O3 AQI',
    'SO2 Mean (ppb)', 'SO2 AQI',
    'CO Mean (ppm)', 'CO AQI',
    'O3_AQI_imp', 'SO2_AQI_imp', 'CO_AQI_imp', 'NO2_AQI_imp'
]

# Aggregate
poll_agg = (
    merged.groupby(["County", "State", "Year"])[pollutant_cols]
          .mean()
          .reset_index()
)

# take the first row per group for the demographic columns
demo_cols = [
    col for col in merged.columns
    if col not in pollutant_cols
    and col not in ["Date Local", "Quarter", "DayOfWeek", "City", "City_County_State"]
    and col not in ["County", "State"]   # <-- critical fix
]


demo_static = (
    merged.groupby(["County", "State"])[demo_cols]
          .first()
          .reset_index()
)

# merge back the demographic static data
final_df = poll_agg.merge(
    demo_static,
    on=["County", "State"],
    how="left"
)

final_df.head()


,County,State,Year_x,NO2 Mean (ppb),NO2 AQI,O3 Mean (ppm),O3 AQI,SO2 Mean (ppb),SO2 AQI,CO Mean (ppm),...,Population.Population per Square Mile,Sales.Accommodation and Food Services Sales,Sales.Retail Sales,Employment.Firms.Total,Employment.Firms.Women-Owned,Employment.Firms.Men-Owned,Employment.Firms.Minority-Owned,Employment.Firms.Nonminority-Owned,Employment.Firms.Veteran-Owned,Employment.Firms.Nonveteran-Owned
0,Ada,ID,2009,6.337568,22.000000,0.033784,44.810811,0.221081,1.108108,0.141622,...,372.8,763099.0,5766679.0,41789.0,14661.0,19409.0,3099.0,36701.0,3803.0,35132.0
1,Ada,ID,2010,9.396750,24.027778,0.026389,33.865278,0.323194,0.692521,0.179785,...,372.8,763099.0,5766679.0,41789.0,14661.0,19409.0,3099.0,36701.0,3803.0,35132.0
2,Ada,ID,2011,9.925652,16.913043,0.012609,20.913043,0.715870,1.956522,0.271957,...,372.8,763099.0,5766679.0,41789.0,14661.0,19409.0,3099.0,36701.0,3803.0,35132.0
3,Adair,OK,2008,2.924057,4.684426,0.030779,36.106557,0.597008,1.406504,0.163934,...,39.6,7219.0,123351.0,1864.0,618.0,978.0,770.0,968.0,109.0,1722.0
4,Adair,OK,2009,3.106545,5.230337,0.028596,32.449438,0.625857,1.623596,0.165843,...,39.6,7219.0,123351.0,1864.0,618.0,978.0,770.0,968.0,109.0,1722.0


# ARCHIVED CELLS FOR REUSE OR DELETION BEFORE FINAL PUBLISHING

In [3]:
df=pd.read_csv('cleaned_pollution_data_with_state_codes.csv')
df.head()

,Unnamed: 0,State,Date Local,NO2 Mean (ppb),NO2 1st Max Value (ppb),NO2 1st Max Hour,NO2 AQI,O3 Mean (ppm),O3 1st Max Value (ppm),O3 1st Max Hour,...,SO2_AQI_imp,CO_AQI_imp,NO2_AQI_imp,Year,Quarter,DayOfWeek,City_County_State,City,AreaType,StateCode
0,0,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,12.86,47.73,17.96,2000,1,Saturday,"Phoenix, Maricopa, Arizona",Phoenix,Urban,AZ
1,0,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,12.86,47.73,17.96,2000,1,Saturday,"Phoenix, Maricopa, Arizona",Phoenix,Urban,AZ
2,1,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,12.86,25.00,17.96,2000,1,Saturday,"Phoenix, Maricopa, Arizona",Phoenix,Urban,AZ
3,1,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,12.86,25.00,17.96,2000,1,Saturday,"Phoenix, Maricopa, Arizona",Phoenix,Urban,AZ
4,2,Arizona,2000-01-01,19.04,49.0,19,46,0.02,0.04,10,...,9.43,47.73,17.96,2000,1,Saturday,"Phoenix, Maricopa, Arizona",Phoenix,Urban,AZ


There are two analyses that we will conduct here:
- feature importance: understand what are the factors in common with the highest levels of pollution, in order to target those, and
- clustering: identify locations with common profiles in order to enable common strategies to be developed that can be deployed across similar areas (economies of scale).

Because this is to focus on how we might develop strategies to address current pollution problems, we will remove the data prior to 2010, as this is the point at which most significant improvements had already taken place. So we will focus on the newer data to establish clusters that are relevant today.

In [4]:
# Step 1: Filter to 2010 onwards
df_recent = df[(df['Year'] >= 2010) & (df['State'] != 'Country Of Mexico')]

# Step 2: Aggregate at City_County_State level
region_pollution = (
    df_recent.groupby('City_County_State')[['NO2 AQI', 'SO2 AQI', 'CO AQI']]
        .mean()
        .reset_index()
)

print("Aggregated dataset shape:", region_pollution.shape)
region_pollution.head()

Aggregated dataset shape: (120, 4)


,City_County_State,NO2 AQI,SO2 AQI,CO AQI
0,"Albuquerque, Bernalillo, New Mexico",24.479102,1.179121,3.482323
1,"Alexandria, Alexandria City, Virginia",23.226040,3.121157,3.197107
2,"Arden-Arcade, Sacramento, California",16.241756,0.917286,5.112323
3,"Austin, Travis, Texas",12.488479,1.005530,1.674033
4,"Baton Rouge, East Baton Rouge, Louisiana",22.579182,4.708510,3.463153


Next, lets see what our optimal mumber of clusters is, by seeing how 'tight' each cluster is (Inertia, smaller numbers indicate tighter clusters) and silhouette (distance from other clusters, higher number indicates distinct separation). This will tell us what number of clusters would seem optimal.

In [5]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Scale features
scaler = StandardScaler()
scaled = scaler.fit_transform(region_pollution[['NO2 AQI', 'SO2 AQI', 'CO AQI']])

# Loop through k values
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(scaled)
    inertia = kmeans.inertia_
    sil_score = silhouette_score(scaled, labels)
    print(f"k={k}, Inertia={inertia:.2f}, Silhouette={sil_score:.3f}")

k=2, Inertia=218.39, Silhouette=0.330
k=3, Inertia=169.72, Silhouette=0.307
k=4, Inertia=105.66, Silhouette=0.374
k=5, Inertia=90.51, Silhouette=0.344
k=6, Inertia=84.33, Silhouette=0.331
k=7, Inertia=73.66, Silhouette=0.285
k=8, Inertia=63.58, Silhouette=0.281
k=9, Inertia=57.73, Silhouette=0.302
k=10, Inertia=50.99, Silhouette=0.324


From this, 4 clusters has the highest Silhouette score, which is the more balanced of the two measures, and this is also the point where the drop in Inertia' starts to ease off. Therefore 4 would be our clearer set of clusters.

In [6]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Step 3: Scale features
scaler = StandardScaler()
scaled = scaler.fit_transform(region_pollution[['NO2 AQI', 'SO2 AQI', 'CO AQI']])

# Step 4: Apply KMeans clustering
kmeans = KMeans(n_clusters=4, random_state=42)
region_pollution['Cluster'] = kmeans.fit_predict(scaled)

# Step 5: Cluster profiles
cluster_profiles = (
    region_pollution.groupby('Cluster')[['NO2 AQI', 'SO2 AQI', 'CO AQI']]
    .mean()
    .round(2)
)

# Step 6 - calculate the average levels for the whole dataset
average_levels = region_pollution[['NO2 AQI', 'SO2 AQI', 'CO AQI']].mean().round(2)
cluster_profiles.loc['Overall Average'] = average_levels

# Step 7; display cluster profile values and dataset average values in a single table. Aldo display the counts of regions in each cluster.
cluster_counts = region_pollution['Cluster'].value_counts().sort_index()
cluster_profiles['Region Count'] = cluster_counts   
print(cluster_profiles)


                 NO2 AQI  SO2 AQI  CO AQI  Region Count
Cluster                                                
0                  21.91     2.25    4.42          47.0
1                   9.10     1.86    2.34          41.0
2                  32.05     3.40    7.72          15.0
3                  20.96     9.98    4.67          17.0
Overall Average    18.67     3.35    4.16           NaN


For our 4 clusters, we want to see what their differences are i.e. how each cluster differs from the overall mean for the dataset. The different gas levels would indicate different strategies for dealing with them.

In [7]:
# Step 6: Overall mean across all regions
overall_mean = region_pollution[['NO2 AQI', 'SO2 AQI', 'CO AQI']].mean().round(2)

# Step 7: Add deviation columns
cluster_profiles['NO2 Δ'] = (cluster_profiles['NO2 AQI'] - overall_mean['NO2 AQI']).round(2)
cluster_profiles['SO2 Δ'] = (cluster_profiles['SO2 AQI'] - overall_mean['SO2 AQI']).round(2)
cluster_profiles['CO Δ'] = (cluster_profiles['CO AQI'] - overall_mean['CO AQI']).round(2)

print("Cluster Profiles with Deviations:")
print(cluster_profiles)

Cluster Profiles with Deviations:
                 NO2 AQI  SO2 AQI  CO AQI  Region Count  NO2 Δ  SO2 Δ  CO Δ
Cluster                                                                    
0                  21.91     2.25    4.42          47.0   3.24  -1.10  0.26
1                   9.10     1.86    2.34          41.0  -9.57  -1.49 -1.82
2                  32.05     3.40    7.72          15.0  13.38   0.05  3.56
3                  20.96     9.98    4.67          17.0   2.29   6.63  0.51
Overall Average    18.67     3.35    4.16           NaN   0.00   0.00  0.00


To understand the cause/significance of the pollution data, we need a greater understanding of how these profiles relate to the area they relate to - population, industry, weather, traffic, as these are all interrelated. To do that, we need to find and join further data tables.

In [9]:
#import population data from parent folderand create a population dataframe
population_df = pd.read_csv('../county_demographics.csv')
population_df.head

<bound method NDFrame.head of                 County State  Age.Percent 65 and Older  \
0     Abbeville County    SC                      22.4   
1        Acadia Parish    LA                      15.8   
2      Accomack County    VA                      24.6   
3           Ada County    ID                      14.9   
4         Adair County    IA                      23.0   
...                ...   ...                       ...   
3134       Yuma County    AZ                      19.3   
3135       Yuma County    CO                      18.7   
3136     Zapata County    TX                      13.2   
3137     Zavala County    TX                      14.6   
3138    Ziebach County    SD                       9.6   

      Age.Percent Under 18 Years  Age.Percent Under 5 Years  \
0                           19.8                        4.7   
1                           25.8                        6.9   
2                           20.7                        5.6   
3                    